# Exercise 6: compare three models on real non-TATA promoters

Use the same **human non-TATA promoter benchmark** as Exercise 3.
Compare **logistic regression**, a **random forest**, and **gradient boosting**, all with **k=6**.

Logistic regression is a linear classifier with a sigmoid output, similar to a single artificial neuron.
It is not the classic perceptron: logistic regression learns with log loss rather than the perceptron update rule.
A random forest averages independently trained trees; gradient boosting adds shallow trees sequentially.

Keep the data split and k-mer representation identical so we can compare model behavior.
We use fixed settings, not a hyperparameter search or an equally tuned competition.

This notebook downloads its data directly; no CSV upload or earlier notebook is needed.
Stop before the final cell during validation discussion: **Run all includes test evaluation**.


## Install the Parquet reader

Run this cell once. Colab already includes NumPy, pandas, scikit-learn, and Matplotlib.
For a local environment, install the dependencies in `course_code/requirements.txt`.


In [ ]:
%pip install -q pyarrow


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

seed = 42
k = 6
threshold = 0.5


## 1. Download the official training and test sets

Label 1 is a non-TATA promoter; label 0 is a non-promoter. Sequences are 251 nucleotides long.
Read the same fixed dataset revision as Exercise 3, then rename `seq` to `sequence`.
Keep the official test set unchanged: do not combine it with training sequences or reshuffle it.

One official test sequence contains the unknown nucleotide `N`. Retain it.
A k-mer absent from the training-fitted vocabulary, including one containing `N`, is ignored during transformation.

Source: the Genomic Benchmarks maintainers' `Genomic_Benchmarks_human_nontata_promoters` dataset on Hugging Face.
This is benchmark evaluation, not chromosome-held-out evaluation or proof of a biological promoter mechanism.


In [ ]:
data_url = (
    "https://huggingface.co/datasets/katarinagresova/"
    "Genomic_Benchmarks_human_nontata_promoters/resolve/"
    "e0003669df0a180c8570ebe091ed91f20fa080ec/data/"
)
benchmark_train = pd.read_parquet(data_url + "train-00000-of-00001-aeb55728756a44ae.parquet")
test_data = pd.read_parquet(data_url + "test-00000-of-00001-d7614d9116c62f44.parquet")

benchmark_train = benchmark_train.rename(columns={"seq": "sequence"})
test_data = test_data.rename(columns={"seq": "sequence"})

assert benchmark_train["sequence"].str.fullmatch("[ACGTN]{251}").all()
assert test_data["sequence"].str.fullmatch("[ACGTN]{251}").all()
assert set(benchmark_train["label"]) == {0, 1}
assert set(test_data["label"]) == {0, 1}

print("Official training sequences:", len(benchmark_train))
print("Official test sequences (reserved):", len(test_data))
print("Official training class counts:")
print(benchmark_train["label"].value_counts())
benchmark_train.head()


## 2. Reserve validation sequences from the official training set

Sample 20% per label from the official training set for validation. Use the remaining 80% for training.
This gives **21,678 training / 5,419 validation / 9,034 official test sequences**.

This is the same split as Exercise 3, not the artificial dataset's 70/20/10 split.
All three models use the same split. Fit the vocabulary and models on training sequences only.


In [ ]:
validation_data = benchmark_train.groupby("label").sample(frac=0.20, random_state=seed)
train_data = benchmark_train.drop(validation_data.index)

y_train = train_data["label"]
y_validation = validation_data["label"]

print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Official test (reserved):", len(test_data))
print("Validation class counts:")
print(validation_data["label"].value_counts())


## 3. Count overlapping 6-mers once

Move a six-nucleotide window one base at a time. A 251-nt sequence contributes **246 overlapping windows**.
For A/C/G/T there are at most **4,096 possible 6-mers**.
Count these windows, ignoring position, without TF-IDF or merging reverse complements.

Fit one vocabulary on training sequences and give all three models the same sparse count matrices.
Do not convert the matrices to dense arrays.


In [ ]:
vectorizer = CountVectorizer(
    analyzer="char", ngram_range=(k, k), lowercase=False, dtype=np.int32,
)
x_train = vectorizer.fit_transform(train_data["sequence"])
x_validation = vectorizer.transform(validation_data["sequence"])

print("Training count matrix:", x_train.shape)
print("Validation count matrix:", x_validation.shape)


## 4. Fit three models with fixed settings

- **Logistic regression:** same settings as Exercise 3, including L2 regularization with `C=1.0`.
- **Random forest:** 100 trees, using available CPU cores, as in Exercise 4.
- **Gradient boosting:** 200 depth-2 trees, learning rate 0.1, and binary log loss, using Exercise 5's boosting budget.

There is no search over k or tree count in this exercise. These settings are fixed before looking at results.
The boosting model is sequential and can take longer to train; wait for each model's completion message.


In [ ]:
models = {
    "Logistic regression": LogisticRegression(
        C=1.0, solver="liblinear", max_iter=1000, random_state=seed,
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=100, random_state=seed, n_jobs=-1,
    ),
    "Gradient boosting": GradientBoostingClassifier(
        loss="log_loss", n_estimators=200, learning_rate=0.1,
        max_depth=2, random_state=seed,
    ),
}

for name, model in models.items():
    print("Training:", name, flush=True)
    model.fit(x_train, y_train)
    print("Finished:", name, flush=True)


## 5. Compare validation performance

Compare all eight familiar metrics. Accuracy, balanced accuracy, precision, recall, and F1 use threshold 0.5.
ROC-AUC and average precision summarize rankings across thresholds. Lower log loss is better; higher is better for the other metrics.

Overlay ROC and precision-recall curves for direct comparison, then inspect each confusion matrix.
The dashed lines show chance-level ROC and the positive-class fraction for precision-recall.
Discuss which model performs best and whether the metrics agree, without opening the test results yet.


In [ ]:
validation_probabilities = {}
rows = []

for name, model in models.items():
    probabilities = model.predict_proba(x_validation)[:, 1]
    validation_probabilities[name] = probabilities
    predictions = (probabilities >= threshold).astype(int)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_validation, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_validation, predictions),
        "precision": precision_score(y_validation, predictions, zero_division=0),
        "recall": recall_score(y_validation, predictions, zero_division=0),
        "f1": f1_score(y_validation, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_validation, probabilities),
        "average_precision": average_precision_score(y_validation, probabilities),
        "log_loss": log_loss(y_validation, probabilities),
    })

validation_results = pd.DataFrame(rows).set_index("model")
print("Validation metrics at k=6:")
print(validation_results.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, probabilities in validation_probabilities.items():
    RocCurveDisplay.from_predictions(y_validation, probabilities, ax=axes[0], name=name)
    PrecisionRecallDisplay.from_predictions(y_validation, probabilities, ax=axes[1], name=name)

axes[0].plot([0, 1], [0, 1], "k--")
axes[1].axhline(y_validation.mean(), color="black", linestyle="--")
axes[0].set_title("Validation ROC, k=6")
axes[1].set_title("Validation precision-recall, k=6")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, probabilities) in zip(axes, validation_probabilities.items()):
    predictions = (probabilities >= threshold).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_validation, predictions, labels=[0, 1], display_labels=["Non-promoter", "Promoter"],
        ax=ax, colorbar=False, cmap="Blues",
        im_kw={"vmin": 0, "vmax": y_validation.value_counts().max()},
    )
    ax.set_title(f"{name}: validation")
plt.tight_layout()
plt.show()


## 6. Final comparison on the official test set

Stop here until all settings are fixed. Evaluate all three already-fitted models once on the same official test set.
Do not refit on validation data, tune settings from test results, or choose a new model using the test results.
This is the final report of the predefined three-model comparison.

**Run all includes this test evaluation.**


In [ ]:
y_test = test_data["label"]
x_test = vectorizer.transform(test_data["sequence"])

test_probabilities = {}
rows = []

for name, model in models.items():
    probabilities = model.predict_proba(x_test)[:, 1]
    test_probabilities[name] = probabilities
    predictions = (probabilities >= threshold).astype(int)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
        "average_precision": average_precision_score(y_test, probabilities),
        "log_loss": log_loss(y_test, probabilities),
    })

test_results = pd.DataFrame(rows).set_index("model")
print("Test metrics at k=6:")
print(test_results.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, probabilities in test_probabilities.items():
    RocCurveDisplay.from_predictions(y_test, probabilities, ax=axes[0], name=name)
    PrecisionRecallDisplay.from_predictions(y_test, probabilities, ax=axes[1], name=name)

axes[0].plot([0, 1], [0, 1], "k--")
axes[1].axhline(y_test.mean(), color="black", linestyle="--")
axes[0].set_title("Test ROC, k=6")
axes[1].set_title("Test precision-recall, k=6")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, probabilities) in zip(axes, test_probabilities.items()):
    predictions = (probabilities >= threshold).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions, labels=[0, 1], display_labels=["Non-promoter", "Promoter"],
        ax=ax, colorbar=False, cmap="Blues",
        im_kw={"vmin": 0, "vmax": y_test.value_counts().max()},
    )
    ax.set_title(f"{name}: test")
plt.tight_layout()
plt.show()
